# 16S Analyses of the Longitudinal Acne Study
## Alpha Diversity Plots

Date last updated: 1/7/25

Notebook authors: Yang Chen, Britta De Pessemier, and Tyler Myers

This notebook plots the following:

- 16S V1-V3 and V4 Shannon alpha diversity plots at ASV level from rarefied abundance tables
- 16S V1-V3 and V4 Faith PD alpha diversity plots at ASV level from rarefied abundance tables

In [34]:
# Import Python packages
import re
import pandas as pd
import numpy as np
import biom
from biom import Table
from biom.util import biom_open
from biom import load_table
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
import os
from matplotlib.colors import ListedColormap
from matplotlib.colors import to_rgba
from skbio.diversity import alpha_diversity
from skbio.diversity.alpha import shannon
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from skbio import TreeNode
from skbio.diversity.alpha import faith_pd
import statsmodels.formula.api as smf
from scipy import stats

In [35]:
# Load the metadata
metadata_path = '../Metadata/metadata_final_22102024.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,subject_ID_x_group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_308_Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_310_Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL,PP_305_Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L,PP_306_Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,moderate,moderate Acne_L,PP_306_Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L,low,low Acne_L,PP_317_Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_14_Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L,low,low Acne_L,PP_314_Acne_L


In [36]:
# Define paths to tables
biom_paths = {
    'V1-V3': '../Data/16S/Tables/intersection_samples_V1V3_V4/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed_16S-aligned.biom',
    'V4': '../Data/16S/Tables/intersection_samples_V1V3_V4/174951_feature-table_16S_V4_rare-3769_sampleIDfixed_16S-aligned.biom'
}

In [37]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum
def load_biom_table(biom_path, metadata_path):
    # Load metadata as a DataFrame from the file path
    metadata = pd.read_csv(metadata_path, sep='\t')

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])
    
    return df

In [38]:
# Use sample samples from V1-V3 and V4 for alpha diversity analyses
def get_clean_sample_ids(biom_path):
    table = load_table(biom_path)
    sample_ids = pd.Series(list(table.ids(axis='sample')))

    sample_ids = (
        sample_ids
        .astype(str)
        .str.replace(r'^.*?(?=LAMI)', '', regex=True)
    )

    return set(sample_ids)

In [39]:
# Subset tables for paired-sample consistency
biom_v1v3 = biom_paths['V1-V3']
biom_v4   = biom_paths['V4']

samples_v1v3 = get_clean_sample_ids(biom_v1v3)
samples_v4   = get_clean_sample_ids(biom_v4)

shared_samples = samples_v1v3.intersection(samples_v4)

print(f"Shared samples between V1–V3 and V4: {len(shared_samples)}")

Shared samples between V1–V3 and V4: 184


In [40]:
# Load metadata
metadata = pd.read_csv(metadata_path, sep='\t')

# Ensure SampleID is string
metadata['SampleID'] = metadata['SampleID'].astype(str)

# Keep only samples shared between V1–V3 and V4
md_shared = metadata[metadata['SampleID'].isin(shared_samples)]

# Count per group
group_counts = (
    md_shared
    .groupby('group')
    .size()
    .reset_index(name='n')
)

print(group_counts)

     group    n
0   Acne_L  123
1  Acne_NL   42
2  Healthy   19


## Shannon Alpha Diversity

In [41]:
def calculate_shannon_alpha_diversity_and_plot(
    outdir,
    biom_path,
    metadata_path,
    group_col='group',
    subject_col='subject_ID',
    title_suffix='',
    keep_samples=None,
    save_biom_path=None
):
    """
    Compute Shannon alpha diversity from a BIOM table, fit a mixed-effects model,
    compute pairwise contrasts, and plot results using LMM p-values.
    """

    def mixedlm_pairwise_pvalues(df, response, group_col, subject_col):
        model = smf.mixedlm(
            f"{response} ~ {group_col}",
            data=df,
            groups=df[subject_col]
        )
        result = model.fit()

        params = result.params
        cov = result.cov_params()
        param_names = params.index.tolist()

        def make_contrast(terms):
            c = np.zeros(len(params))
            for term, weight in terms.items():
                c[param_names.index(term)] = weight
            return c

        contrasts = {
            "Healthy vs Acne_NL": make_contrast({f"{group_col}[T.Acne_NL]": 1}),
            "Healthy vs Acne_L":  make_contrast({f"{group_col}[T.Acne_L]": 1}),
            "Acne_NL vs Acne_L":  make_contrast({
                f"{group_col}[T.Acne_L]": 1,
                f"{group_col}[T.Acne_NL]": -1
            })
        }

        rows = []
        for name, c in contrasts.items():
            est = np.dot(c, params.values)
            se = np.sqrt(np.dot(c, np.dot(cov.values, c)))
            z  = est / se
            p  = 2 * (1 - stats.norm.cdf(abs(z)))

            rows.append({
                "comparison": name,
                "estimate": est,
                "z": z,
                "p": p
            })

        return pd.DataFrame(rows), result

    table = load_table(biom_path)
    metadata = pd.read_csv(metadata_path, sep="\t")

    if keep_samples is not None:
        keep_samples = set(keep_samples)
        samples_in_table = {
            sid for sid in table.ids(axis="sample")
            if re.sub(r"^.*?(?=LAMI)", "", str(sid)) in keep_samples
        }
        table = table.filter(samples_in_table, axis="sample", inplace=False)

    if save_biom_path is not None:
        os.makedirs(os.path.dirname(save_biom_path), exist_ok=True)
        with h5py.File(save_biom_path, "w") as f:
            table.to_hdf5(f, "alpha-diversity-input")

    shannon_df = pd.DataFrame({
        "sample_id": list(table.ids(axis="sample")),
        "Shannon": [shannon(table.data(sid, axis="sample"))
                    for sid in table.ids(axis="sample")]
    })

    shannon_df["sample_id"] = (
        shannon_df["sample_id"]
        .astype(str)
        .str.replace(r"^.*?(?=LAMI)", "", regex=True)
    )

    shannon_df.to_csv(f'../Analyses/16S/diversity/Shannon_{key}.csv')

    metadata["SampleID"] = metadata["SampleID"].astype(str)
    metadata = metadata.merge(
        shannon_df,
        left_on="SampleID",
        right_on="sample_id",
        how="inner"
    )

    metadata = metadata.dropna(subset=[group_col, subject_col, "Shannon"])

    metadata[group_col] = pd.Categorical(
        metadata[group_col],
        categories=["Healthy", "Acne_NL", "Acne_L"],
        ordered=False
    )

    pairwise_df, lmm_result = mixedlm_pairwise_pvalues(
        metadata, "Shannon", group_col, subject_col
    )

    print(f"\n[INFO] MixedLM results ({title_suffix})")
    print(lmm_result.summary())
    print(pairwise_df)

    box_palette = {
        "Healthy": "#3333B3",
        "Acne_NL": "#5cbccb",
        "Acne_L": "#f16c52"
    }

    darker_palette = {
        "Healthy": "#23238f",
        "Acne_NL": "#44a7b6"
    }

    severity_palette = {
        "low": "#f3b7a6",
        "moderate": "#e07b63",
        "high": "#b7372a"
    }

    fig, ax = plt.subplots(figsize=(4, 6))

    sns.boxplot(
        data=metadata,
        x=group_col,
        y="Shannon",
        order=["Healthy", "Acne_NL", "Acne_L"],
        palette=box_palette,
        showcaps=True,
        showfliers=False,
        boxprops=dict(edgecolor="black", linewidth=1.2),
        whiskerprops=dict(color="black", linewidth=1.2),
        medianprops=dict(color="black", linewidth=1.5),
        ax=ax
    )

    non_lesional = metadata[metadata[group_col] != "Acne_L"]

    sns.stripplot(
        data=non_lesional,
        x=group_col,
        y="Shannon",
        order=["Healthy", "Acne_NL", "Acne_L"],
        hue=group_col,
        hue_order=["Healthy", "Acne_NL"],
        palette=darker_palette,
        jitter=True,
        dodge=False,
        size=4,
        alpha=0.9,
        edgecolor="black",
        linewidth=0.4,
        ax=ax,
        zorder=2
    )

    ax.legend_.remove()

    if "severity_level" in metadata.columns:
        acne_l = metadata[metadata[group_col] == "Acne_L"]

        sns.stripplot(
            data=acne_l,
            x=group_col,
            y="Shannon",
            order=["Healthy", "Acne_NL", "Acne_L"],
            hue="severity_level",
            hue_order=["low", "moderate", "high"],
            palette=severity_palette,
            jitter=True,
            dodge=False,
            size=4,
            alpha=0.9,
            edgecolor="black",
            linewidth=0.4,
            ax=ax,
            zorder=3
        )

        handles, labels = ax.get_legend_handles_labels()

        label_map = {
            "low": "Low (1–2)",
            "moderate": "Mod (3–4)",
            "high": "High (5–6)"
        }

        severity_handles = []
        severity_labels = []

        for h, l in zip(handles, labels):
            if l in label_map:
                severity_handles.append(h)
                severity_labels.append(label_map[l])

        for h in severity_handles:
            h.set_sizes([16])
            h.set_edgecolor("black")
            h.set_linewidth(0.6)

        if title_suffix == "V1-V3":
            ax.legend(
                severity_handles,
                severity_labels,
                title="Severity",
                fontsize=8,
                title_fontsize=9,
                frameon=True,
                loc="upper right",
                bbox_to_anchor=(1.0, 0.85)
            )
        elif title_suffix == "V4":
            ax.legend(
                severity_handles,
                severity_labels,
                title="Severity",
                fontsize=8,
                title_fontsize=9,
                frameon=True,
                loc="lower left"
            )

    def add_pvalue(ax, x1, x2, y, p):
        stars = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""
        label = f"{stars}  p={p:.2g}".strip()
        ax.plot([x1, x1, x2, x2], [y, y + 0.05, y + 0.05, y], lw=1.2, c="black")
        ax.text((x1 + x2) / 2, y + 0.06, label,
                ha="center", va="bottom", fontsize=8)

    ymax = metadata["Shannon"].max()
    step = ymax * 0.12
    current_y = ymax + step

    comparison_map = {
        "Healthy vs Acne_NL": ("Healthy", "Acne_NL"),
        "Healthy vs Acne_L":  ("Healthy", "Acne_L"),
        "Acne_NL vs Acne_L":  ("Acne_NL", "Acne_L")
    }

    for _, row in pairwise_df.iterrows():
        g1, g2 = comparison_map[row["comparison"]]
        x1 = ["Healthy", "Acne_NL", "Acne_L"].index(g1)
        x2 = ["Healthy", "Acne_NL", "Acne_L"].index(g2)
        add_pvalue(ax, x1, x2, current_y, row["p"])
        current_y += step * 0.8

    group_counts = metadata[group_col].value_counts().to_dict()

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels([
        f"H\n(n={group_counts.get('Healthy', 0)})",
        f"ANL\n(n={group_counts.get('Acne_NL', 0)})",
        f"AL\n(n={group_counts.get('Acne_L', 0)})"
    ])

    ax.set_title(f"16S rRNA ({title_suffix}) Alpha Diversity")
    ax.set_xlabel("")
    ax.set_ylabel("Shannon Index")

    sns.despine()
    plt.tight_layout()

    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(
        outdir,
        f"Shannon_alpha_diversity_{title_suffix.replace(' ', '_')}.png"
    )
    plt.savefig(outpath, dpi=600, bbox_inches="tight")
    plt.close(fig)


In [42]:
# Plot Shannon Diversity plots for both V1-V3 and V4
for key, biom_path in biom_paths.items():
    calculate_shannon_alpha_diversity_and_plot(
        outdir='../Figures/Supplementary/Suppl_Figure_3/',
        biom_path=biom_path,
        metadata_path=metadata_path,
        group_col='group',
        title_suffix=key,
        keep_samples=shared_samples
    )


[INFO] MixedLM results (V1-V3)
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Shannon  
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.1288   
Min. group size:    1        Log-Likelihood:      -101.9997
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         0.971    0.269  3.610 0.000  0.444  1.498
group[T.Acne_NL] -0.414    0.347 -1.193 0.233 -1.093  0.266
group[T.Acne_L]  -0.264    0.344 -0.768 0.443 -0.938  0.410
Group Var         0.448    0.508                           

           comparison  estimate         z         p
0  Healthy vs Acne_NL -0.413767 -1.193279  0.232760
1   Healthy vs Acne_L -0.264002 -0

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na


[INFO] MixedLM results (V4)
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Shannon  
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.4812   
Min. group size:    1        Log-Likelihood:      -219.0704
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         3.865    0.452  8.546 0.000  2.978  4.751
group[T.Acne_NL] -1.360    0.581 -2.342 0.019 -2.499 -0.222
group[T.Acne_L]  -1.268    0.574 -2.208 0.027 -2.394 -0.143
Group Var         1.214    0.711                           

           comparison  estimate         z         p
0  Healthy vs Acne_NL -1.360435 -2.341625  0.019200
1   Healthy vs Acne_L -1.268448 -2.20

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na

## Faith Phylogenetic Alpha Diversity

In [43]:
def calculate_faithpd_and_plot(
    biom_path,
    tree_path,
    metadata_path,
    group_col,
    subject_col,
    title_suffix,
    outdir,
    key
):
    """
    Compute Faith PD alpha diversity, fit a linear mixed-effects model,
    compute pairwise contrasts, and plot results using LMM p-values.
    """

    def mixedlm_pairwise_pvalues(df, response, group_col, subject_col):
        model = smf.mixedlm(
            f"{response} ~ {group_col}",
            data=df,
            groups=df[subject_col]
        )
        result = model.fit()

        params = result.params
        cov = result.cov_params()
        names = params.index.tolist()

        def make_contrast(weights):
            c = np.zeros(len(params))
            for term, w in weights.items():
                c[names.index(term)] = w
            return c

        contrasts = {
            "Healthy vs Acne_NL": make_contrast({f"{group_col}[T.Acne_NL]": 1}),
            "Healthy vs Acne_L":  make_contrast({f"{group_col}[T.Acne_L]": 1}),
            "Acne_NL vs Acne_L":  make_contrast({
                f"{group_col}[T.Acne_L]": 1,
                f"{group_col}[T.Acne_NL]": -1
            })
        }

        rows = []
        for name, c in contrasts.items():
            est = np.dot(c, params.values)
            se = np.sqrt(np.dot(c, np.dot(cov.values, c)))
            z  = est / se
            p  = 2 * (1 - stats.norm.cdf(abs(z)))
            rows.append(dict(comparison=name, p=p))

        return pd.DataFrame(rows), result

    table = load_table(biom_path)
    tree = TreeNode.read(tree_path)

    counts = table.to_dataframe(dense=True).T

    faith = {
        sid: faith_pd(
            counts.loc[sid].values,
            counts.columns,
            tree
        )
        for sid in counts.index
    }

    faith_df = (
        pd.DataFrame.from_dict(faith, orient="index", columns=["Faith_PD"])
        .reset_index()
        .rename(columns={"index": "SampleID"})
    )

    faith_df["SampleID"] = (
        faith_df["SampleID"]
        .astype(str)
        .str.replace(r"^.*?(?=LAMI)", "", regex=True)
    )

    faith_df.to_csv(
        f"../Analyses/16S/diversity/FaithPD_{key}.csv",
        index=False
    )

    metadata = pd.read_csv(metadata_path, sep="\t")
    metadata["SampleID"] = (
        metadata["SampleID"]
        .astype(str)
        .str.replace(r"^.*?(?=LAMI)", "", regex=True)
    )

    metadata = metadata.merge(faith_df, on="SampleID", how="inner")
    metadata = metadata.dropna(subset=[group_col, subject_col, "Faith_PD"])

    metadata[group_col] = pd.Categorical(
        metadata[group_col],
        categories=["Healthy", "Acne_NL", "Acne_L"],
        ordered=False
    )

    metadata["severity_category"] = pd.cut(
        metadata["local_lesion_severity"],
        bins=[0, 2, 4, 6],
        labels=["Low", "Mod", "High"],
        include_lowest=True
    )

    plot_order = ["Healthy", "Acne_NL", "Acne_L"]
    plot_order = [g for g in plot_order if g in metadata[group_col].unique()]

    pairwise_df, lmm_result = mixedlm_pairwise_pvalues(
        metadata, "Faith_PD", group_col, subject_col
    )

    print(f"\n[INFO] Faith PD MixedLM results ({key})")
    print(lmm_result.summary())
    print(pairwise_df)

    palette = {
        "Healthy": "#23238f",
        "Acne_NL": "#44a7b6",
        "Acne_L": "#d8543e"
    }

    severity_palette = {
        "Low": "#F1948A",
        "Mod": "#EC7063",
        "High": "#C0392B"
    }

    fig, ax = plt.subplots(figsize=(3.5, 6))

    sns.boxplot(
        data=metadata,
        x=group_col,
        y="Faith_PD",
        order=plot_order,
        palette=palette,
        fliersize=0,
        linewidth=1.2,
        ax=ax
    )

    sns.stripplot(
        data=metadata[metadata[group_col] != "Acne_L"],
        x=group_col,
        y="Faith_PD",
        order=plot_order,
        hue=group_col,
        hue_order=["Healthy", "Acne_NL"],
        palette=palette,
        jitter=True,
        dodge=False,
        size=5,
        edgecolor="black",
        linewidth=0.6,
        ax=ax
    )

    ax.legend_.remove()

    acne_l = metadata[metadata[group_col] == "Acne_L"]
    if not acne_l.empty:
        sns.stripplot(
            data=acne_l,
            x=group_col,
            y="Faith_PD",
            order=plot_order,
            hue="severity_category",
            hue_order=["Low", "Mod", "High"],
            palette=severity_palette,
            jitter=True,
            dodge=False,
            size=5,
            edgecolor="black",
            linewidth=0.6,
            ax=ax
        )

        handles, labels = ax.get_legend_handles_labels()
        handles = handles[-3:]
        labels = ["Low (1–2)", "Mod (3–4)", "High (5–6)"]

        legend_pos = {
            "V1-V3": dict(loc="upper right", bbox_to_anchor=(0.91, 0.87)),
            "V4":    dict(loc="upper left",  bbox_to_anchor=(0.03, 0.2)),
        }

        ax.legend(
            handles,
            labels,
            title="Severity",
            fontsize=8,
            title_fontsize=9,
            frameon=True,
            **legend_pos.get(key, {})
        )

    ymax = metadata["Faith_PD"].max()
    step = (metadata["Faith_PD"].max() - metadata["Faith_PD"].min()) * 0.08
    current_y = ymax + step

    comparison_map = {
        "Healthy vs Acne_NL": ("Healthy", "Acne_NL"),
        "Healthy vs Acne_L":  ("Healthy", "Acne_L"),
        "Acne_NL vs Acne_L":  ("Acne_NL", "Acne_L")
    }

    for _, row in pairwise_df.iterrows():
        g1, g2 = comparison_map[row["comparison"]]
        x1 = plot_order.index(g1)
        x2 = plot_order.index(g2)

        p = row["p"]
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        label = f"{stars} p={p:.3g}".strip()

        ax.plot(
            [x1, x1, x2, x2],
            [current_y, current_y + step * 0.2,
             current_y + step * 0.2, current_y],
            lw=1,
            c="black"
        )

        ax.text(
            (x1 + x2) / 2,
            current_y + step * 0.25,
            label,
            ha="center",
            va="bottom",
            fontsize=9
        )

        current_y += step * 1.1

    label_map = {"Healthy": "H", "Acne_NL": "ANL", "Acne_L": "AL"}
    counts = metadata[group_col].value_counts().to_dict()

    ax.set_xticklabels([
        f"{label_map[g]}\n(n={counts.get(g, 0)})"
        for g in plot_order
    ])

    ax.set_xlabel("")
    ax.set_ylabel("Faith PD")
    ax.set_title(f"16S rRNA ({title_suffix}) Alpha Diversity")

    sns.despine()
    plt.tight_layout()

    os.makedirs(outdir, exist_ok=True)
    plt.savefig(
        os.path.join(outdir, f"FaithPD_alpha_diversity_{key}.png"),
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(fig)


In [44]:
# Define inputs
tree_paths = {
    "V1-V3": "../Data/16S/Trees/179426_V1V3_ASVs_seqIDs.rooted.nwk",
    "V4":    "../Data/16S/Trees/174951_V4_ASVs_seqIDs.rooted.nwk"
}

outdir = "../Figures/Main/Figure_3/"


In [45]:
# Run Faith PD for V1-V3
calculate_faithpd_and_plot(
    biom_path=biom_paths["V1-V3"],
    tree_path=tree_paths["V1-V3"],
    metadata_path=metadata_path,
    group_col="group",
    title_suffix="V1–V3",
    outdir=outdir,
    key="V1-V3",
    subject_col="subject_ID"
)



[INFO] Faith PD MixedLM results (V1-V3)
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Faith_PD 
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.4334   
Min. group size:    1        Log-Likelihood:      -206.4811
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         4.378    0.355 12.323 0.000  3.681  5.074
group[T.Acne_NL] -1.077    0.454 -2.375 0.018 -1.967 -0.188
group[T.Acne_L]  -0.678    0.446 -1.521 0.128 -1.553  0.196
Group Var         0.691    0.457                           

           comparison         p
0  Healthy vs Acne_NL  0.017550
1   Healthy vs Acne_L  0.128268
2   Acne_NL vs Acne_L  0.000740


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na

In [46]:
# Run Faith PD for V4
calculate_faithpd_and_plot(
    biom_path=biom_paths["V4"],
    tree_path=tree_paths["V4"],
    metadata_path=metadata_path,
    group_col="group",
    title_suffix="V4",
    outdir=outdir,
    key="V4",
    subject_col="subject_ID"
)



[INFO] Faith PD MixedLM results (V4)
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Faith_PD 
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.9440   
Min. group size:    1        Log-Likelihood:      -282.7966
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         9.854    0.752 13.109 0.000  8.381 11.327
group[T.Acne_NL] -1.563    0.970 -1.612 0.107 -3.463  0.338
group[T.Acne_L]  -1.306    0.962 -1.358 0.175 -3.191  0.580
Group Var         3.525    1.418                           

           comparison         p
0  Healthy vs Acne_NL  0.107062
1   Healthy vs Acne_L  0.174612
2   Acne_NL vs Acne_L  0.141152


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na